In [1]:
import numpy as np
import matplotlib.pyplot as plt


def c_factor(n):
    """
    Average path length of unsuccessful search in a Binary Search Tree (BST)
    given n nodes. Used to normalize Isolation Tree path lengths.
    """
    if n <= 1:
        return 0.0
    if n == 2:
        return 1.0
    # Euler's constant harmonic number approximation: H(n-1)
    harmonic = np.log(n - 1.0) + 0.5772156649
    return 2.0 * harmonic - (2.0 * (n - 1.0) / n)


class IsolationTree:
    """Single Random Partitioning Tree (iTree)."""

    def __init__(self, height_limit):
        self.height_limit = height_limit
        self.split_feat = None
        self.split_val = None
        self.left = None
        self.right = None
        self.size = 0
        self.is_external = False

    def fit(self, X, current_height=0):
        n_samples, n_features = X.shape
        self.size = n_samples

        # Stopping conditions: max height reached, <= 1 sample, or all identical values
        if current_height >= self.height_limit or n_samples <= 1:
            self.is_external = True
            return self

        # Randomly select a feature that has variance
        valid_feats = [f for f in range(n_features) if np.min(X[:, f]) < np.max(X[:, f])]
        if not valid_feats:
            self.is_external = True
            return self

        self.split_feat = np.random.choice(valid_feats)
        f_min = np.min(X[:, self.split_feat])
        f_max = np.max(X[:, self.split_feat])

        # Randomly pick a threshold between min and max
        self.split_val = np.random.uniform(f_min, f_max)

        left_mask = X[:, self.split_feat] < self.split_val
        right_mask = ~left_mask

        self.left = IsolationTree(self.height_limit).fit(X[left_mask], current_height + 1)
        self.right = IsolationTree(self.height_limit).fit(X[right_mask], current_height + 1)
        return self

    def path_length(self, x, current_height=0):
        if self.is_external:
            return current_height + c_factor(self.size)

        if x[self.split_feat] < self.split_val:
            return self.left.path_length(x, current_height + 1)
        else:
            return self.right.path_length(x, current_height + 1)


class IsolationForestScratch:
    """Ensemble of Isolation Trees for Unsupervised Anomaly Detection."""

    def __init__(self, n_estimators=100, max_samples=256, contamination=0.05):
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.contamination = contamination
        self.trees = []
        self.psi = 0
        self.threshold_ = 0.5

    def fit(self, X):
        n_samples = X.shape[0]
        self.psi = min(self.max_samples, n_samples)
        height_limit = int(np.ceil(np.log2(self.psi)))

        self.trees = []
        for _ in range(self.n_estimators):
            idx = np.random.choice(n_samples, self.psi, replace=False)
            X_sub = X[idx]
            tree = IsolationTree(height_limit=height_limit).fit(X_sub)
            self.trees.append(tree)

        # Calibrate anomaly score threshold using contamination ratio
        scores = self.anomaly_score(X)
        self.threshold_ = np.percentile(scores, 100.0 * (1.0 - self.contamination))
        return self

    def anomaly_score(self, X):
        """Returns normalized anomaly score in [0, 1]. Higher = more anomalous."""
        n_samples = X.shape[0]
        avg_heights = np.zeros(n_samples)

        for i in range(n_samples):
            total_h = sum(tree.path_length(X[i]) for tree in self.trees)
            avg_heights[i] = total_h / len(self.trees)

        c_psi = c_factor(self.psi)
        # Score formula: s(x, psi) = 2 ^ (-E(h(x)) / c(psi))
        scores = np.power(2.0, -avg_heights / c_psi)
        return scores

    def predict(self, X):
        """Returns -1 for outliers and +1 for normal inliers."""
        scores = self.anomaly_score(X)
        return np.where(scores >= self.threshold_, -1, 1)


# =========================================================================
# DEMO EXECUTION
# =========================================================================
if __name__ == "__main__":
    np.random.seed(42)

    # 1. Generate normal dense clusters + extreme scattered anomalies
    inliers_1 = np.random.randn(150, 2) * 0.6 + np.array([2.0, 2.0])
    inliers_2 = np.random.randn(150, 2) * 0.6 + np.array([-2.0, -2.0])
    outliers = np.random.uniform(low=-6.0, high=6.0, size=(20, 2))
    X = np.vstack([inliers_1, inliers_2, outliers])

    # 2. Fit Isolation Forest from Scratch
    iforest = IsolationForestScratch(n_estimators=100, max_samples=128, contamination=0.06)
    iforest.fit(X)
    scores = iforest.anomaly_score(X)
    preds = iforest.predict(X)

    print("=" * 65)
    print("  ISOLATION FOREST ANOMALY DETECTION FROM SCRATCH (NUMPY)")
    print("=" * 65) 
    print(f"Total Samples Analyzed      : {X.shape[0]}")
    print(f"Sub-sample Size (psi)       : {iforest.psi}")
    print(f"Calibrated Score Threshold   : {iforest.threshold_:.4f}")
    print(f"Detected Inliers (+1)        : {np.sum(preds == 1)}")
    print(f"Detected Anomalies (-1)      : {np.sum(preds == -1)}")
    print("=" * 65)

  ISOLATION FOREST ANOMALY DETECTION FROM SCRATCH (NUMPY)
Total Samples Analyzed       : 320
Sub-sample Size (psi)        : 128
Calibrated Score Threshold   : 0.5582
Detected Inliers (+1)        : 300
Detected Anomalies (-1)      : 20
